# Segment Contribution Analysis


## Load data

1. Aggregate transaction-level data.
To make the analysis more meaningful, I introduced controlled perturbations to simulate noticeable growth patterns.

2. Select a target merchant and benchmark key business metrics (e.g., transaction amount, transaction count) against peer merchants within the same category.

In [11]:
import pandas as pd
import numpy as np
import etl
from exhaustive_segment_search import exhaustive_eval
from config import DIMENSION_COLS
from helper import grpby_dim_val

In [12]:
TARGET_MERCHANTS = "fraud_Kilback LLC"
target_col = "amt_growth_ctc_diff"
dim_cols = DIMENSION_COLS

df_combined = etl.load_rca_data(TARGET_MERCHANTS)

File exists, loading data from /Users/yanxinye/github/competency-intelligence/src/segment_analysis/etl/rca_fraud_Kilback LLC.csv...


## Segment-Level Decomposition

### Option1: Exhaustive Search of All Dimension Interactions

An intuitive approach to identifying segments that explain the gap between a target merchant and its peers is to compute metrics across all combinations of dimension columns (i.e., k-way interactions).

In [13]:
sorted_res = exhaustive_eval(df_combined, target_col, dim_cols=dim_cols)

total = df_combined[target_col].sum()
print(f"The total growth difference is: {total:.2%}")
print("Top segment drivers:\n")
for i, (k, v) in enumerate(sorted_res[:20], 1):
    print(f"{i:>2}. {k:<50} | {v:>8.2%} | contribution: {v/total:>6.2%}")

The total growth difference is: 25.87%
Top segment drivers:

 1. gender = F                                         |   18.82% | contribution: 72.73%
 2. generation = Millennial, market = South            |   14.53% | contribution: 56.17%
 3. generation = Gen X                                 |   14.10% | contribution: 54.49%
 4. market = South                                     |   14.01% | contribution: 54.16%
 5. category = food_dining                             |   13.58% | contribution: 52.49%
 6. category = grocery_pos                             |   12.29% | contribution: 47.51%
 7. generation = Millennial, market = South, category = grocery_pos |   12.24% | contribution: 47.29%
 8. generation = Millennial                            |   11.14% | contribution: 43.07%
 9. generation = Millennial, market = South, gender = F, category = grocery_pos |    9.83% | contribution: 38.00%
10. generation = Gen X, market = West                  |    9.68% | contribution: 37.42%
11. gender 


This results in **O(2ⁿ)** complexity, with computational cost growing exponentially as the number of dimensions increases.

To maintain tractability, we can cap the interaction depth (e.g., *k ≤ 3*), trading off computational efficiency against the ability to capture higher-order interactions.

### Option2: Tree search


Total amt_growth_ctc_diff: 25.87%



In [52]:
left_node

,generation,state,market,gender,category,amt_growth_ctc_diff
0,Boomer,AL,South,F,food_dining,0.003273
1,Boomer,AL,South,F,grocery_pos,-0.003385
4,Boomer,AR,South,F,food_dining,-0.000788
5,Boomer,AR,South,F,grocery_pos,-0.002013
8,Boomer,AZ,West,F,food_dining,0.005554
...,...,...,...,...,...,...
680,Unknown,WI,Midwest,F,grocery_pos,0.002617
682,Unknown,WV,South,F,food_dining,-0.000598
683,Unknown,WV,South,F,grocery_pos,-0.001406
686,Unknown,WY,West,F,food_dining,-0.001829


In [53]:
right_node

,generation,state,market,gender,category,amt_growth_ctc_diff
2,Boomer,AL,South,M,food_dining,-0.002267
3,Boomer,AL,South,M,grocery_pos,0.009058
6,Boomer,AR,South,M,food_dining,-0.000058
7,Boomer,AR,South,M,grocery_pos,0.000203
10,Boomer,AZ,West,M,food_dining,0.001103
...,...,...,...,...,...,...
673,Unknown,TX,South,M,food_dining,-0.000234
674,Unknown,TX,South,M,grocery_pos,0.003717
681,Unknown,WI,Midwest,M,grocery_pos,0.000000
684,Unknown,WV,South,M,food_dining,0.000378


In [34]:
# sorted_res